# Robustness to Stochastic Missing Mechanisms

In this notebook, we test notMIWAE's robustness when the missing mechanism is **probabilistic** rather than deterministic. Instead of "x > mean → always missing", we have "x > mean → probability p of being missing".

This is a more realistic scenario and tests whether notMIWAE can extract the signal from noisy observations of the missing pattern.

## Run this if you work on Colab

In [ ]:
# Run if destination doesn't already exists
!git clone https://github.com/VincentGefflaut/notMIWAE-project.git

In [ ]:
# Always run before working in Colab
import sys
sys.path.insert(0,'/content/notMIWAE-project')
!cd notMIWAE-project; git pull

## Helper functions

In [18]:
import numpy as np
import pandas as pd
import tensorflow.compat.v1 as tf
# tf.disable_eager_execution() # Déjà géré par notre patch sed, mais sécurité
import os
import sys
import matplotlib.pyplot as plt

# Imports du repo local
from MIWAE import MIWAE
from notMIWAE import notMIWAE
import trainer
import utils

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# --- Fonctions de chargement et préparation ---

def load_uci_data(dataset_name):
    """Charge les datasets via les URLs UCI."""
    print(f"Chargement de {dataset_name}...")
    if dataset_name == 'White':
        url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv"
        data = pd.read_csv(url, sep=';').values
    elif dataset_name == 'Red':
        url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
        data = pd.read_csv(url, sep=';').values
    elif dataset_name == 'Banknote':
        url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00267/data_banknote_authentication.txt"
        data = pd.read_csv(url, header=None).values
    elif dataset_name == 'Concrete':
        # Concrete nécessite souvent openpyxl
        url = "https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls"
        try:
            data = pd.read_excel(url).values
        except:
            print("Erreur chargement Excel Concrete. Installation dépendance...")
            os.system('pip install xlrd')
            data = pd.read_excel(url).values
    else:
        raise ValueError("Dataset non supporté dans ce script demo.")
    return data

def introduce_mnar_missing(X, missing_mechanism = lambda x: x > np.mean(x), missing_col_ratio=0.5):
    """
    Simule le mécanisme MNAR (Self-masking) décrit dans l'article.
    Le mécanisme de masquage correspond à la fonction missing_mechanism passée en argument.
    Par défaut, si la valeur > moyenne de la colonne, elle devient manquante.
    Appliqué sur les 100*missing_col_ratio pourcent premières colonnes.
    """
    N, D = X.shape
    Xnan = X.copy()
    n_missing_cols = int(D * missing_col_ratio)

    for j in range(n_missing_cols):
        col = Xnan[:, j]
        mask_condition = missing_mechanism(col)
        Xnan[mask_condition, j] = np.nan
    # Version avec 0 à la place des NaN (pour l'input réseau)
    Xz = Xnan.copy()
    Xz[np.isnan(Xnan)] = 0

    return Xnan, Xz

## Core experiment

In [15]:
def probabilistic_mask(X, p=0.8):
    """
    Probabilistic self-masking: if x > mean(x), it has probability p of being missing.

    Args:
        X: Column data
        p: Probability of being missing when x > mean
    """
    meets_condition = X > np.mean(X)
    random_draws = np.random.random(len(X))
    return meets_condition & (random_draws < p)

In [ ]:
# Complete experiment: all missing models across all p values
MAX_ITER = 1000
BATCH_SIZE = 16
N_SAMPLES = 20
L_IMP = 500
VAL_SPLIT = 0.2

DATASET = 'Banknote'
P_VALUES = [0.5, 0.7, 0.9, 1.0]
MISSING_MODELS = ['MIWAE', 'notMIWAE_selfmasking', 'notMIWAE_linear', 'notMIWAE_nonlinear']

results_complete = []

for p_val in P_VALUES:
    print(f"\n{'='*80}")
    print(f"Testing with p = {p_val}")
    print(f"{'='*80}\n")

    # Prepare data
    data = load_uci_data(DATASET)
    data = data[:, :-1]
    scaler = StandardScaler()
    data = scaler.fit_transform(data)

    np.random.seed(42)
    perm = np.random.permutation(len(data))
    data = data[perm, :]

    data_train, data_val = train_test_split(data, test_size=VAL_SPLIT, random_state=42, shuffle=True)

    # Create probabilistic mask
    mask_fn = lambda x: probabilistic_mask(x, p=p_val)

    np.random.seed(42)
    Xnan_train, Xz_train = introduce_mnar_missing(data_train, missing_mechanism=mask_fn)
    S_train = np.array(~np.isnan(Xnan_train), dtype=np.float32)

    np.random.seed(43)
    Xnan_val, Xz_val = introduce_mnar_missing(data_val, missing_mechanism=mask_fn)
    S_val = np.array(~np.isnan(Xnan_val), dtype=np.float32)

    missing_rate = 1 - np.mean(S_train)
    print(f"Missing rate: {missing_rate:.2%}\n")

    # Test each missing model
    for model_name in MISSING_MODELS:
        tf.reset_default_graph()
        sess_name = f"/tmp/{DATASET}_{model_name}_p{p_val}"

        print(f"\n--- Modèle: {model_name} ---")

        if model_name == 'MIWAE':
            model = MIWAE(Xnan_train, Xnan_val, n_latent=data.shape[1]-1, n_samples=N_SAMPLES,
                            n_hidden=128, name=sess_name)
            history = trainer.train(model, batch_size=BATCH_SIZE, max_iter=MAX_ITER, name=sess_name)

            rmse_train = utils.imputationRMSE(model, data_train, Xz_train, Xnan_train, S_train, L_IMP)[0]
            rmse_val = utils.imputationRMSE(model, data_val, Xz_val, Xnan_val, S_val, L_IMP)[0]

        elif 'notMIWAE' in model_name:
            proc = 'selfmasking'
            model = notMIWAE(Xnan_train, Xnan_val, n_latent=data.shape[1]-1, n_samples=N_SAMPLES,
                                n_hidden=128, missing_process=proc, name=sess_name)
            history = trainer.train(model, batch_size=BATCH_SIZE, max_iter=MAX_ITER, name=sess_name)

            rmse_train = utils.not_imputationRMSE(model, data_train, Xz_train, Xnan_train, S_train, L_IMP)[0]
            rmse_val = utils.not_imputationRMSE(model, data_val, Xz_val, Xnan_val, S_val, L_IMP)[0]

        print(f"Val RMSE: {rmse_val:.4f}")

        results_complete.append({
            'p': p_val,
            'Model': model_name.replace('notMIWAE_selfmasking', 'notMIWAE'),
            'RMSE_val': rmse_val
        })

print("\n" + "="*80)
print("=== FINAL RESULTS: All Missing Models vs All p Values ===")
print("="*80)
df_complete = pd.DataFrame(results_complete)
print("\n" + df_complete.to_string(index=False))


Testing with p = 0.01

Chargement de Banknote...


HTTPError: HTTP Error 502: Bad Gateway

In [ ]:
# Prepare data for plotting
df_miwae = df_complete[df_complete['Model'] == 'MIWAE']
df_selfmasking = df_complete[df_complete['Missing Model'] == 'notMIWAE_selfmasking']
df_linear = df_complete[df_complete['Missing Model'] == 'notMIWAE_linear']
df_nonlinear = df_complete[df_complete['Missing Model'] == 'notMIWAE_nonlinear']

plt.figure(figsize=(12, 7))

# Plot each model
plt.plot(df_miwae['p'], df_miwae['RMSE_val'], 'd-',
         label='MIWAE (baseline)', linewidth=2, markersize=10, color='blue', linestyle='--')
plt.plot(df_selfmasking['p'], df_selfmasking['RMSE_val'], 's-',
         label='notMIWAE (selfmasking)', linewidth=2, markersize=10, color='orange')
plt.plot(df_linear['p'], df_linear['RMSE_val'], 'o-',
         label='notMIWAE (linear)', linewidth=2, markersize=10, color='green')
plt.plot(df_nonlinear['p'], df_nonlinear['RMSE_val'], '^-',
         label='notMIWAE (nonlinear)', linewidth=2, markersize=10, color='red')

plt.xlabel('Probability p (of missing when x > mean)', fontsize=13, fontweight='bold')
plt.ylabel('Validation RMSE', fontsize=13, fontweight='bold')
plt.title('Missing Model Flexibility vs Stochastic Noise - Banknote Dataset',
          fontsize=15, fontweight='bold')
plt.legend(fontsize=11, loc='best')
plt.grid(True, alpha=0.3, linestyle='--')
plt.xticks(P_VALUES, fontsize=11)
plt.yticks(fontsize=11)

# Add annotations
plt.axhline(y=0.6, color='gray', linestyle=':', alpha=0.5)
plt.text(0.52, 0.1, 'Very noisy\n(~MAR)', ha='center', fontsize=9,
         style='italic', color='gray', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
plt.text(1.0, 0.1, 'Deterministic\n(pure MNAR)', ha='center', fontsize=9,
         style='italic', color='gray', bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))

plt.tight_layout()
plt.show()